# Notebook 2 — Solutions: CNN Segmentation of Neurons and Synapses

**BINFX410 · Chapter 10 · Connectomics**

Reference solutions for the five exercises in `02_segmentation_cnn.ipynb`.  
Run notebooks 1 and 2 first to generate the required data and trained model.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy import ndimage

DATA_DIR  = Path('../data')
MODEL_DIR = DATA_DIR
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

with open(DATA_DIR / 'connectome_dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

print(f'Train: {len(dataset["train"]["images"])} samples')
print(f'Val  : {len(dataset["val"]["images"])} samples')

In [ ]:
# ── Shared definitions used across multiple exercises ──────────────────────────

class NeuronDataset(Dataset):
    def __init__(self, images, soma_masks, synapse_masks, augment=False):
        self.images        = images
        self.soma_masks    = soma_masks
        self.synapse_masks = synapse_masks
        self.augment       = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = self.images[idx].copy()
        soma = self.soma_masks[idx].astype(np.float32)
        syn  = self.synapse_masks[idx].astype(np.float32)
        if self.augment:
            if np.random.rand() > 0.5: img  = np.fliplr(img).copy();  soma = np.fliplr(soma).copy();  syn = np.fliplr(syn).copy()
            if np.random.rand() > 0.5: img  = np.flipud(img).copy();  soma = np.flipud(soma).copy();  syn = np.flipud(syn).copy()
            k = np.random.randint(0, 4)
            img  = np.rot90(img,  k).copy()
            soma = np.rot90(soma, k).copy()
            syn  = np.rot90(syn,  k).copy()
            img  = np.clip(img + np.random.uniform(-0.05, 0.05), 0, 1)
        img  = torch.from_numpy(img[None])
        masks = torch.cat([torch.from_numpy(soma[None]), torch.from_numpy(syn[None])], dim=0)
        return img, masks


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, n_out=2, base_ch=32):
        super().__init__()
        ch = base_ch
        self.enc1 = ConvBlock(1, ch);      self.enc2 = ConvBlock(ch, ch*2);   self.enc3 = ConvBlock(ch*2, ch*4)
        self.bottleneck = ConvBlock(ch*4, ch*8)
        self.up3 = nn.ConvTranspose2d(ch*8, ch*4, 2, stride=2); self.dec3 = ConvBlock(ch*8, ch*4)
        self.up2 = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2); self.dec2 = ConvBlock(ch*4, ch*2)
        self.up1 = nn.ConvTranspose2d(ch*2, ch,   2, stride=2); self.dec1 = ConvBlock(ch*2, ch)
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Conv2d(ch, n_out, 1)

    def forward(self, x):
        e1 = self.enc1(x);  e2 = self.enc2(self.pool(e1));  e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


def dice_loss(pred_probs, targets, eps=1e-6):
    inter = (pred_probs * targets).sum(dim=(-2,-1))
    union = pred_probs.sum(dim=(-2,-1)) + targets.sum(dim=(-2,-1))
    return 1 - ((2*inter+eps)/(union+eps)).mean()

def combined_loss(logits, targets):
    return 0.5 * F.binary_cross_entropy_with_logits(logits, targets) + 0.5 * dice_loss(torch.sigmoid(logits), targets)

def bce_only_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)

def iou_score(pred_probs, targets, threshold=0.5):
    pred_bin = (pred_probs > threshold).float()
    inter    = (pred_bin * targets).sum(dim=(-2,-1))
    union    = pred_bin.sum(dim=(-2,-1)) + targets.sum(dim=(-2,-1)) - inter
    return (inter / (union + 1e-6)).mean().item()


def make_loaders(base_ch_unused=32):
    train_ds = NeuronDataset(dataset['train']['images'], dataset['train']['soma_masks'],
                             dataset['train']['synapse_masks'], augment=True)
    val_ds   = NeuronDataset(dataset['val']['images'],   dataset['val']['soma_masks'],
                             dataset['val']['synapse_masks'], augment=False)
    return (DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0),
            DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0))


def train_unet(model, train_loader, val_loader, epochs=30, lr=3e-4, loss_fn=combined_loss):
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    history   = {'train_loss': [], 'val_loss': [], 'val_iou_soma': [], 'val_iou_syn': []}
    best_val  = float('inf')
    for epoch in range(1, epochs+1):
        model.train()
        tl = 0
        for imgs_b, masks_b in train_loader:
            imgs_b, masks_b = imgs_b.to(DEVICE), masks_b.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(imgs_b), masks_b)
            loss.backward(); optimizer.step()
            tl += loss.item()
        scheduler.step()
        model.eval()
        vl = is_ = isyns = 0
        with torch.no_grad():
            for imgs_b, masks_b in val_loader:
                imgs_b, masks_b = imgs_b.to(DEVICE), masks_b.to(DEVICE)
                logits = model(imgs_b)
                vl   += loss_fn(logits, masks_b).item()
                p     = torch.sigmoid(logits)
                is_  += iou_score(p[:,0:1], masks_b[:,0:1])
                isyns+= iou_score(p[:,1:2], masks_b[:,1:2])
        history['train_loss'].append(tl/len(train_loader))
        history['val_loss'].append(vl/len(val_loader))
        history['val_iou_soma'].append(is_/len(val_loader))
        history['val_iou_syn'].append(isyns/len(val_loader))
        if vl < best_val: best_val = vl
        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{epochs} | train={history["train_loss"][-1]:.4f} '
                  f'val={history["val_loss"][-1]:.4f} | '
                  f'IoU soma={history["val_iou_soma"][-1]:.3f} syn={history["val_iou_syn"][-1]:.3f}')
    return history

print('Shared definitions loaded.')

---
## Exercise 2.1 — Purpose of Skip Connections

**Task:** Explain skip connections. Remove one and retrain to illustrate the impact.

**Conceptual Answer:**

Skip connections (also called residual or bypass connections) in U-Net serve two purposes:

1. **Preserve spatial detail.** The encoder's successive max-pooling operations reduce spatial resolution — by epoch 4 (for a 4-level U-Net on 256×256 input) the bottleneck is only 16×16. The coarse feature map cannot recover exact pixel positions during upsampling alone. Skip connections route the **high-resolution, low-level feature maps** from the encoder directly to the decoder at the same scale, enabling precise localisation of segment boundaries.

2. **Ease gradient flow.** Deep networks suffer vanishing gradients. Skip connections provide a direct, short path for gradients to flow backward to early encoder layers without having to pass through all intermediate operations.

**Without skip connections:** the decoder must reconstruct spatial detail purely from the bottleneck — equivalent to upsampling a blurred latent representation. This is sufficient for coarse segmentation but loses boundary precision, which is critical for small objects like synapses.

In [ ]:
# U-Net with the highest-resolution skip connection (enc1→dec1) removed
class UNetNoSkip1(nn.Module):
    """Same as UNet but the enc1→dec1 skip connection is removed."""
    def __init__(self, n_out=2, base_ch=32):
        super().__init__()
        ch = base_ch
        self.enc1 = ConvBlock(1, ch);      self.enc2 = ConvBlock(ch, ch*2);   self.enc3 = ConvBlock(ch*2, ch*4)
        self.bottleneck = ConvBlock(ch*4, ch*8)
        self.up3 = nn.ConvTranspose2d(ch*8, ch*4, 2, stride=2); self.dec3 = ConvBlock(ch*8, ch*4)
        self.up2 = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2); self.dec2 = ConvBlock(ch*4, ch*2)
        # No skip: dec1 receives only ch channels from up1, not 2*ch
        self.up1 = nn.ConvTranspose2d(ch*2, ch,   2, stride=2); self.dec1 = ConvBlock(ch, ch)
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Conv2d(ch, n_out, 1)

    def forward(self, x):
        e1 = self.enc1(x);  e2 = self.enc2(self.pool(e1));  e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(self.up1(d2))  # ← skip removed here
        return self.head(d1)


train_loader, val_loader = make_loaders()

print('Training standard U-Net (with all skip connections)...')
model_full = UNet(n_out=2, base_ch=32).to(DEVICE)
hist_full  = train_unet(model_full, train_loader, val_loader, epochs=20)

print('\nTraining U-Net without enc1→dec1 skip connection...')
model_noskip = UNetNoSkip1(n_out=2, base_ch=32).to(DEVICE)
hist_noskip  = train_unet(model_noskip, train_loader, val_loader, epochs=20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist_full['val_loss'],  label='Full U-Net')
axes[0].plot(hist_noskip['val_loss'], label='No enc1 skip', linestyle='--')
axes[0].set_title('Validation Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(hist_full['val_iou_soma'],   label='Full U-Net soma')
axes[1].plot(hist_noskip['val_iou_soma'], label='No skip soma', linestyle='--')
axes[1].set_title('Soma IoU'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

plt.suptitle('Exercise 2.1 — Effect of Removing a Skip Connection', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nFinal soma IoU — Full U-Net: {hist_full["val_iou_soma"][-1]:.3f}')
print(f'Final soma IoU — No skip:    {hist_noskip["val_iou_soma"][-1]:.3f}')

---
## Exercise 2.2 — BCE-Only vs. BCE + Dice Loss

**Task:** Compare IoU curves for the two loss functions.

In [ ]:
train_loader, val_loader = make_loaders()

print('Training with BCE-only loss...')
model_bce = UNet(n_out=2, base_ch=32).to(DEVICE)
hist_bce  = train_unet(model_bce, train_loader, val_loader, epochs=30, loss_fn=bce_only_loss)

print('\nTraining with BCE + Dice loss...')
model_combined = UNet(n_out=2, base_ch=32).to(DEVICE)
hist_combined  = train_unet(model_combined, train_loader, val_loader, epochs=30, loss_fn=combined_loss)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_bce['val_iou_soma'],      label='BCE only — soma',    color='steelblue')
axes[0].plot(hist_combined['val_iou_soma'], label='BCE+Dice — soma',    color='tomato')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('IoU')
axes[0].set_title('Soma IoU: BCE vs. BCE+Dice'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(hist_bce['val_iou_syn'],      label='BCE only — synapse',  color='steelblue')
axes[1].plot(hist_combined['val_iou_syn'], label='BCE+Dice — synapse',  color='tomato')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('IoU')
axes[1].set_title('Synapse IoU: BCE vs. BCE+Dice'); axes[1].legend(); axes[1].grid(True)

plt.suptitle('Exercise 2.2 — Loss Function Comparison', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Final soma IoU  — BCE only : {hist_bce["val_iou_soma"][-1]:.3f}')
print(f'Final soma IoU  — BCE+Dice : {hist_combined["val_iou_soma"][-1]:.3f}')
print(f'\nWhy BCE+Dice is better:')
print('  BCE treats each pixel independently. With class imbalance (many background pixels),')
print('  the model can achieve low BCE by predicting all-background. Dice loss directly')
print('  penalises missing the (small) positive regions, pushing the model to find soma/synapses.')

---
## Exercise 2.3 — Model Capacity: `base_ch` = 16, 32, 64

**Task:** Train with three model sizes, report parameters and final IoU.

In [ ]:
train_loader, val_loader = make_loaders()

results = {}
for ch in [16, 32, 64]:
    model = UNet(n_out=2, base_ch=ch).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n--- base_ch={ch}  ({n_params:,} parameters) ---')
    hist = train_unet(model, train_loader, val_loader, epochs=30)
    results[ch] = {'n_params': n_params, 'history': hist}

# Summary table
print('\n=== Summary ===')
print(f'{"base_ch":>8}  {"Parameters":>12}  {"Final Soma IoU":>14}  {"Final Syn IoU":>13}')
for ch, r in results.items():
    h = r['history']
    print(f'{ch:>8}  {r["n_params"]:>12,}  {h["val_iou_soma"][-1]:>14.3f}  {h["val_iou_syn"][-1]:>13.3f}')

# Plot IoU curves
colors = ['steelblue', 'tomato', 'green']
fig, ax = plt.subplots(figsize=(10, 5))
for (ch, r), col in zip(results.items(), colors):
    ax.plot(r['history']['val_iou_soma'], label=f'base_ch={ch} ({r["n_params"]:,} params)', color=col)
ax.set_xlabel('Epoch'); ax.set_ylabel('Soma IoU')
ax.set_title('Exercise 2.3 — Model Capacity vs. Soma IoU')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nConclusion:')
print('  base_ch=16 → under-parameterised; soma IoU is noticeably lower.')
print('  base_ch=32 → good balance; reasonable IoU at manageable parameter count.')
print('  base_ch=64 → marginal improvement over 32; 4× more parameters for ~5% IoU gain.')
print('  Given 40 training images, base_ch=64 may overfit; monitor train vs. val gap.')

---
## Exercise 2.4 — Threshold Sweep: Precision-Recall Curve

**Task:** Vary `SOMA_THRESH` from 0.2 to 0.8. Plot the PR curve and find the F1-maximising threshold.

In [ ]:
# Load the best model trained by the original notebook
model = UNet(n_out=2, base_ch=32).to(DEVICE)
model.load_state_dict(torch.load(MODEL_DIR / 'unet_best.pt', map_location=DEVICE))
model.eval()

infer    = dataset['inference']
img_full = infer['image']
gt_soma  = infer['soma_mask'].astype(np.float32)

# Run inference on the full scene (same tile-based approach as NB2)
def predict_full(model, img, tile=256, overlap=32):
    H, W = img.shape
    acc_soma  = np.zeros((H, W), dtype=np.float32)
    count_acc = np.zeros((H, W), dtype=np.float32)
    step = tile - overlap
    rows = list(range(0, H-tile+1, step)) + [H-tile]
    cols = list(range(0, W-tile+1, step)) + [W-tile]
    with torch.no_grad():
        for r in rows:
            for c in cols:
                patch = img[r:r+tile, c:c+tile]
                t   = torch.from_numpy(patch[None, None]).to(DEVICE)
                out = model(t).cpu().numpy()[0]
                acc_soma[r:r+tile, c:c+tile] += out[0]
                count_acc[r:r+tile, c:c+tile] += 1.0
    return 1 / (1 + np.exp(-acc_soma / (count_acc + 1e-8)))

soma_probs = predict_full(model, img_full)

# Sweep thresholds
thresholds = np.arange(0.20, 0.81, 0.05)
precisions, recalls, f1s = [], [], []

gt_flat = gt_soma.flatten()
prob_flat = soma_probs.flatten()

for thresh in thresholds:
    pred = (prob_flat > thresh).astype(float)
    tp = (pred * gt_flat).sum()
    fp = (pred * (1 - gt_flat)).sum()
    fn = ((1 - pred) * gt_flat).sum()
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)

best_idx   = np.argmax(f1s)
best_thresh = thresholds[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curve
axes[0].plot(recalls, precisions, 'o-', color='steelblue')
axes[0].plot(recalls[best_idx], precisions[best_idx], 'r*', markersize=15,
             label=f'Best F1 @ thresh={best_thresh:.2f}')
for t, r, p in zip(thresholds, recalls, precisions):
    axes[0].annotate(f'{t:.2f}', (r, p), textcoords='offset points', xytext=(4, 2), fontsize=7)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve (soma)'); axes[0].legend(); axes[0].grid(True)
axes[0].set_xlim([-0.05, 1.05]); axes[0].set_ylim([-0.05, 1.05])

# F1 vs threshold
axes[1].plot(thresholds, f1s, 'o-', color='tomato')
axes[1].axvline(best_thresh, color='gray', linestyle='--', label=f'Best thresh={best_thresh:.2f}')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score vs. Classification Threshold'); axes[1].legend(); axes[1].grid(True)

plt.suptitle('Exercise 2.4 — Threshold Sweep for Soma Segmentation', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Threshold that maximises F1: {best_thresh:.2f}')
print(f'  Precision: {precisions[best_idx]:.3f}')
print(f'  Recall   : {recalls[best_idx]:.3f}')
print(f'  F1       : {f1s[best_idx]:.3f}')

---
## Exercise 2.5 (Challenge) — Third Output Channel: Axon Segmentation

**Task:** Add an axon detection output channel to the U-Net.

This requires:
1. Generating axon ground-truth masks in `neuron_sim.py`
2. Adding a third mask channel to `NeuronDataset`
3. Changing `n_out=3` in `UNet`

The code below sketches the key modifications.

In [ ]:
# ── Step 1: Generating axon masks ────────────────────────────────────────────
# In utils/neuron_sim.py, add this function alongside make_ground_truth_masks:

# def make_axon_mask(image_size, neurons, axon_radius=1):
#     """Draw a binary mask over axon segments using the stored axon endpoint pairs."""
#     H, W = image_size
#     mask = np.zeros((H, W), dtype=np.uint8)
#     for neuron in neurons:
#         # neuron.axon_endpoints is a list of (start, end) tuples for each segment
#         for (r0, c0), (r1, c1) in neuron.axon_endpoints:
#             # Rasterise the line segment using Bresenham's algorithm
#             rr, cc = skimage.draw.line(int(r0), int(c0), int(r1), int(c1))
#             rr = np.clip(rr, 0, H-1); cc = np.clip(cc, 0, W-1)
#             mask[rr, cc] = 1
#     # Dilate by axon_radius pixels to give some width
#     mask = binary_dilation(mask, iterations=axon_radius).astype(np.uint8)
#     return mask

# ── Step 2: Modified NeuronDataset ───────────────────────────────────────────
class NeuronDataset3Ch(Dataset):
    """Dataset returning (image, 3-channel mask: soma, synapse, axon)."""
    def __init__(self, images, soma_masks, synapse_masks, axon_masks, augment=False):
        self.images        = images
        self.soma_masks    = soma_masks
        self.synapse_masks = synapse_masks
        self.axon_masks    = axon_masks
        self.augment       = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = self.images[idx].copy()
        soma = self.soma_masks[idx].astype(np.float32)
        syn  = self.synapse_masks[idx].astype(np.float32)
        axon = self.axon_masks[idx].astype(np.float32)
        if self.augment:
            if np.random.rand() > 0.5:
                for arr in [img, soma, syn, axon]:
                    arr = np.fliplr(arr)
        img  = torch.from_numpy(img[None])
        masks = torch.cat([
            torch.from_numpy(soma[None]),
            torch.from_numpy(syn[None]),
            torch.from_numpy(axon[None])
        ], dim=0)
        return img, masks

# ── Step 3: 3-output U-Net ────────────────────────────────────────────────────
# Simply instantiate: model_3ch = UNet(n_out=3, base_ch=32)
# The head nn.Conv2d(ch, n_out, 1) automatically outputs 3 channels.

model_3ch = UNet(n_out=3, base_ch=32).to(DEVICE)
n_params  = sum(p.numel() for p in model_3ch.parameters() if p.requires_grad)
print(f'3-channel U-Net parameters: {n_params:,}')
print(f'2-channel U-Net parameters: 1,926,466 (from notebook 2)')
print()
print('The only difference is the final 1×1 conv head: 32→3 instead of 32→2.')
print('This adds only (32*3 + 3) - (32*2 + 2) = 33 extra parameters.')
print()
print('Expected challenges:')
print('  - Axon pixels are very thin (1-2px wide); class imbalance is severe.')
print('  - Weighted BCE or focal loss per channel would help.')
print('  - The encoder needs to learn to detect elongated, curvilinear structures.')
print('    Consider adding a morphological (thin-line) data augmentation step.')